# NB4: P1 full vs baseline — resnet32, Inc10
Seeds 0 and 1 on separate T4 lanes: baseline → locked pair-specific preflight → full.
All 10 tasks, final epochs, no warmup or clipping. Attach CIFAR-100, pinned repository source,
and prior notebook Output for resume. NB3/NB4 also require the existing C-Flat BN protocol.json.
Runtime is unverified on T4: the notebook prints measured projections and warns over 11 hours;
it never shortens the task sequence. Source comes directly from the verified pinned Git commit; no ZIP overlay is used.


In [ ]:
REPO_COMMIT='ec93947cd43565aeedd29d7cf85a65d093b6d1b9'
SOURCE_DIGEST='0f6046d85d24073b8d91f1404124964aabc1a10ba24096074fee063443b4acd5'
SOURCE_FILE_HASHES={}
NOTEBOOK='NB4'
BACKBONE='resnet32'
INCREMENT=10
RUN_TAG='p1_compare_nb4'
import time
SESSION_HOURS=11.5
SESSION_START=time.time()
VNA_DEADLINE=SESSION_START+SESSION_HOURS*3600
DISK_CAP_BYTES=20_000_000_000
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
RUN_ROOT = Path("/kaggle/working/p1_compare")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if len(branch) == 40 and all(c in "0123456789abcdef" for c in branch):
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", branch], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", branch], env=env, check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO,):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)

os.environ['VNA_SOURCE_DIGEST']=SOURCE_DIGEST
(RUN_ROOT/'source_manifest.json').write_text(json.dumps(dict(code_commit=REPO_COMMIT,
    source_digest=SOURCE_DIGEST,verification='git_commit_sha256'),indent=2)+'\n')
print('Source: verified pinned Git commit; no ZIP overlay',flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONUNBUFFERED="1", TQDM_DISABLE="1", PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
required_cifar_files = {"train", "test", "meta"}
cifar_dirs = []
for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
    path = Path(directory)
    if (path.name == "cifar-100-python" and
            required_cifar_files.issubset(filenames)):
        cifar_dirs.append(path.parent)
cifar_dirs = sorted(set(cifar_dirs), key=str)
if not cifar_dirs:
    visible = []
    for directory, dirnames, filenames in os.walk(
            "/kaggle/input", followlinks=True):
        visible.append({
            "directory": directory,
            "directories": sorted(dirnames)[:10],
            "files": sorted(filenames)[:10],
        })
        if len(visible) == 20:
            break
    raise FileNotFoundError(
        "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
        f"Visible input inventory: {visible}")
DATA_ROOT = cifar_dirs[0]
print("CIFAR-100 root:", DATA_ROOT)

if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO)]
print("Official TEST verified:", DATA_ROOT / "cifar-100-python/test")



In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_kaggle import restore_stage_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs"
STAGE_B.mkdir(parents=True, exist_ok=True)
ROOT=STAGE_B/RUN_TAG
ROOT.mkdir(parents=True,exist_ok=True)
restore_stage_archives(Path('/kaggle/input'), ROOT, RUN_TAG)
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python", "VNA_DEADLINE": str(VNA_DEADLINE)}
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)
archive_lock = threading.Lock(); archive_stop = threading.Event(); archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs/p1_compare_nb4",), exclude_parts=("data",))
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
archive_errors = []
lane_timings = {}
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.io_utils import atomic_json_save
import fcntl
import hashlib
import math
from experiments.p1_compare import estimate_lane
from methods.vna_v3 import FULL_POLICY_VERSION, POLICY_VERSION

def read(path,default=None):
    try:return json.loads(path.read_text())
    except FileNotFoundError:return default

BN_PROTOCOL=None
if BACKBONE=='resnet32':
    candidates=[]
    for path in Path('/kaggle/input').rglob('protocol.json'):
        try:p=read(path,{})
        except (ValueError,OSError):continue
        if p.get('arm')=='icarl_cflat' and p.get('backbone')=='bn' and not p.get('fake_data'):
            rates=tuple(p.get('training',{}).get(k) for k in ('init_lr','lrate'))
            if all(isinstance(v,(int,float)) and math.isfinite(v) and v>0 for v in rates):candidates.append((path,rates))
    if not candidates:raise FileNotFoundError('Attach the existing C-Flat BN protocol.json; LR will not be guessed')
    if len({rates for _,rates in candidates})!=1:raise ValueError('Conflicting C-Flat BN learning rates')
    BN_PROTOCOL=candidates[0][0]
    print('C-FLAT BN LR SOURCE:',BN_PROTOCOL,'LR:',candidates[0][1],
          'SHA256:',hashlib.sha256(BN_PROTOCOL.read_bytes()).hexdigest(),flush=True)

SCHEDULE=ROOT/'_schedules'/f'{BACKBONE}_inc{INCREMENT}.json'
SCHEDULE.parent.mkdir(parents=True,exist_ok=True)
schedule_lock=threading.Lock()
estimate_lock=threading.Lock()
errors=[]

def command(arm,seed,out):
    cmd=[sys.executable,'-u','-m','experiments.p1_compare','--arm',arm,
        '--seed',str(seed),'--backbone',BACKBONE,'--class-order','superclass',
        '--increment',str(INCREMENT),'--max-tasks',str(100//INCREMENT),
        '--warmup-epochs','0','--gir-rule','schur_dead','--solver','gram','--curvature','power10',
        '--nobn-lr-scale','.5','--device','cuda:0','--workers','2',
        '--data-dir',str(DATA_ROOT),'--out',str(out)]
    if BN_PROTOCOL:cmd+=['--bn-lr-protocol',str(BN_PROTOCOL)]
    return cmd

def schedule(gpu):
    with schedule_lock,SCHEDULE.with_suffix('.lock').open('a') as file:
        fcntl.flock(file,fcntl.LOCK_EX)
        if not SCHEDULE.exists():
            cmd=command('icarl_full',0,ROOT/'_preflight')
            cmd+=['--preflight-only','--schedule-output',str(SCHEDULE),'--schedule-source','self']
            if not launch(cmd,gpu,'preflight',RUN_ROOT/'preflight.log'):return False
            archive_outputs()
        value=read(SCHEDULE,{})
        expected=dict(authoritative=True,backbone=BACKBONE,init_cls=INCREMENT,increment=INCREMENT,
            code_commit=REPO_COMMIT,source_digest=SOURCE_DIGEST,solver='gram',curvature='power10',
            policy_version=FULL_POLICY_VERSION,gir_live=False)
        if any(value.get(k)!=v for k,v in expected.items()) or value.get('synthetic'):
            raise ValueError('K0 authority mismatch')
        print(NOTEBOOK,'K0 =',value['K0'],json.dumps(value),flush=True)
        estimate()
        return True

def estimate():
    authority=read(SCHEDULE,{})
    preflight=read(ROOT/'_preflight'/'summary.json',{})
    result=dict(notebook=NOTEBOOK,lanes=[])
    for seed in (0,1):
        base=ROOT/f'icarl__s{seed}'
        baseline=read(base/'summary.json',None)
        if baseline is None:baseline=read(base/'run'/'summary.json',{})
        if baseline.get('status')=='FAILED_DIVERGED':
            result['lanes'].append(dict(seed=seed,status='NO_FULL_RUNTIME_ESTIMATE_BASELINE_DIVERGED',
                baseline_outcome='FAILED_DIVERGED',diverged=baseline.get('diverged')))
            continue
        if baseline.get('status')!='COMPLETED' or not authority.get('calibration') or 'seconds' not in preflight:
            result['lanes'].append(dict(seed=seed,status='PENDING_MEASURED_BASELINE_OR_PREFLIGHT'));continue
        lane=estimate_lane(INCREMENT,baseline['seconds'],authority,preflight['seconds'])
        lane.update(seed=seed,status='ESTIMATED',
            measured_baseline_ggn_seconds=sum(p['seconds'] for p in read(base/'ggn_power10.json',[])))
        result['lanes'].append(lane)
        if lane['total_hours_range'][1]>11:print('WARNING:',NOTEBOOK,'seed',seed,
            'projects above 11 hours:',lane['total_hours_range'],'sessions:',lane['sessions_11h_range'],
            '; keep all tasks and resume next session',flush=True)
    ready=[lane for lane in result['lanes'] if lane['status']=='ESTIMATED']
    if len(ready)==2:
        result['notebook_hours_range']=[max(lane['total_hours_range'][i] for lane in ready) for i in (0,1)]
        result['notebook_sessions_11h_range']=[max(lane['sessions_11h_range'][i] for lane in ready) for i in (0,1)]
    with estimate_lock:atomic_json_save(result,ROOT/'time_estimate.json')
    print('TIME ESTIMATE:',json.dumps(result),flush=True)

def lane(gpu,seed):
    for arm in ('icarl','icarl_full'):
        if pause_due():return
        name=f'{arm}__s{seed}';out=ROOT/name
        existing=read(out/'summary.json',{})
        if existing.get('status') in ('COMPLETED','FAILED_DIVERGED'):
            p=existing.get('identity',{})
            expected=dict(code_commit=REPO_COMMIT,source_digest=SOURCE_DIGEST,arm=arm,seed=seed,
                backbone=BACKBONE,solver='gram',curvature='power10',fake_data=False,
                expected_tasks=100//INCREMENT,policy_version=FULL_POLICY_VERSION if arm=='icarl_full' else POLICY_VERSION,gir_live=False)
            actual={k:p.get(k) for k in expected}
            expected.update(warmup_epochs=0,grad_clip=None,increment=INCREMENT,memory_size=2000)
            actual.update(warmup_epochs=p.get('warmup_epochs',0),grad_clip=p.get('grad_clip'),
                increment=p.get('cfg',{}).get('increment'),memory_size=p.get('cfg',{}).get('memory_size'))
            mismatches={k:dict(saved=actual[k],required=v) for k,v in expected.items() if actual[k]!=v}
            if mismatches:raise ValueError('Terminal result identity mismatch: '+name+' '+json.dumps(mismatches,sort_keys=True))
            if existing['status']=='COMPLETED' and len(existing.get('tasks',[]))!=100//INCREMENT:
                raise ValueError('Completed result is missing tasks: '+name)
            print(name,'existing terminal status:',existing['status'],flush=True);continue
        try:
            if arm=='icarl_full' and not schedule(gpu):return
            cmd=command(arm,seed,out)
            if arm=='icarl_full':cmd+=['--schedule',str(SCHEDULE)]
            if not launch(cmd,gpu,name,RUN_ROOT/(name+'.log')):
                if pause_due():return
                errors.append(dict(job=name,error='nonzero exit'));continue
            result=read(out/'summary.json',{})
            print(name,'STATUS:',result.get('status'),result.get('diverged'),flush=True)
        except Exception as error:errors.append(dict(job=name,error=str(error)))
        archive_outputs();estimate()

try:
    estimate()
    with ThreadPoolExecutor(max_workers=2) as pool:
        jobs=[pool.submit(lane,0,0),pool.submit(lane,1,1)]
        for job in jobs:job.result()
finally:
    archive_stop.set();archiver.join(timeout=10);estimate();archive_outputs()
if errors:print('JOB ERRORS',json.dumps(errors),flush=True)
